In [2]:
# ==========================================
# Notebook 11
# Hybrid Recommendation Engine
# ==========================================

import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

from scipy.sparse.linalg import svds

In [3]:
profiles_df = pd.read_csv("../data/item_profiles.csv")

profiles_df.head()

,item_id,title,category,tags,content_profile
0,1,The Early Days of Stripe,Startup,startup fintech founders entrepreneurship vent...,Title: The Early Days of Stripe Category: Star...
1,2,Building SpaceX,Startup,startup fintech founders entrepreneurship vent...,Title: Building SpaceX Category: Startup Tags:...
2,3,AI for Healthcare,Artificial Intelligence,ai machine-learning llm deep-learning transfor...,Title: AI for Healthcare Category: Artificial ...
3,4,The Psychology of Habits,Wellness,habits productivity self-improvement,Title: The Psychology of Habits Category: Well...
4,5,Cloud Computing Fundamentals,Technology,cloud computing distributed-systems scalability,Title: Cloud Computing Fundamentals Category: ...


In [4]:
interactions_df = pd.read_csv("../data/user_interactions.csv")

interactions_df.head()

,user_id,item_id,rating
0,101,1,5
1,101,2,5
2,101,5,4
3,102,1,5
4,102,8,5


In [5]:
item_embeddings = np.load("../data/item_embeddings.npy")

item_embeddings.shape

(10, 384)

In [6]:
user_embeddings = np.load("../data/user_embeddings.npy")

user_embeddings.shape

(5, 384)

In [7]:
interaction_matrix = interactions_df.pivot_table(
    index="user_id", columns="item_id", values="rating", fill_value=0
)

interaction_matrix

item_id,1,2,3,4,5,6,7,8,9,10
user_id,,,,,,,,,,
101,5.0,5.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0
102,5.0,4.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0
103,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,5.0,0.0
104,0.0,0.0,5.0,0.0,0.0,0.0,4.0,0.0,0.0,5.0
105,0.0,0.0,0.0,0.0,5.0,5.0,4.0,0.0,0.0,0.0


In [8]:
interaction_matrix.shape

(5, 10)

In [9]:
matrix = interaction_matrix.values

In [10]:
matrix

array([[5., 5., 0., 0., 4., 0., 0., 0., 0., 0.],
       [5., 4., 0., 0., 0., 0., 0., 5., 0., 0.],
       [0., 0., 0., 5., 0., 0., 0., 0., 5., 0.],
       [0., 0., 5., 0., 0., 0., 4., 0., 0., 5.],
       [0., 0., 0., 0., 5., 5., 4., 0., 0., 0.]])

In [11]:
U, sigma, VT = svds(matrix, k=2)

In [12]:
sigma = np.diag(sigma)

In [13]:
predicted_ratings = np.dot(np.dot(U, sigma), VT)

In [14]:
predicted_ratings_df = pd.DataFrame(
    predicted_ratings,
    index=interaction_matrix.index,
    columns=interaction_matrix.columns,
)

predicted_ratings_df

item_id,1,2,3,4,5,6,7,8,9,10
user_id,,,,,,,,,,
101,4.875574e+00,4.400459e+00,-7.882155e-16,2.805968e-17,2.778877e+00,7.788767e-01,6.231013e-01,2.375574e+00,2.805968e-17,-7.882155e-16
102,4.875574e+00,4.375574e+00,-7.788767e-01,8.718040e-17,1.900459e+00,5.494663e-16,-6.231013e-01,2.500000e+00,8.718040e-17,-7.788767e-01
103,1.152401e-16,9.780400e-17,-1.942453e-16,1.540744e-32,-1.533879e-16,-1.758357e-16,-2.960648e-16,8.718040e-17,1.540744e-32,-1.942453e-16
104,-7.788767e-01,-6.231013e-01,2.500000e+00,-1.942453e-16,2.375574e+00,2.375574e+00,3.900459e+00,-7.788767e-01,-1.942453e-16,2.500000e+00
105,7.788767e-01,7.788767e-01,2.375574e+00,-1.758357e-16,3.123101e+00,2.500000e+00,3.900459e+00,-2.431117e-16,-1.758357e-16,2.375574e+00


In [15]:
def collaborative_recommendations(user_id, top_k=10):

    scores = predicted_ratings_df.loc[user_id]

    consumed = interactions_df[interactions_df["user_id"] == user_id][
        "item_id"
    ].tolist()

    scores = scores.drop(consumed)

    scores = scores.sort_values(ascending=False)

    results = []

    for item_id, score in scores.head(top_k).items():

        item = profiles_df[profiles_df["item_id"] == item_id].iloc[0]

        results.append(
            {"item_id": item_id, "title": item["title"], "collaborative_score": score}
        )

    return pd.DataFrame(results)

In [16]:
collaborative_recommendations(101)

,item_id,title,collaborative_score
0,8,Startup Fundraising Guide,2.375574e+00
1,6,Modern Data Engineering,7.788767e-01
2,7,Machine Learning in Finance,6.231013e-01
3,4,The Psychology of Habits,2.805968e-17
4,9,Nutrition Science,2.805968e-17
5,3,AI for Healthcare,-7.882155e-16
6,10,Large Language Models,-7.882155e-16


In [17]:
def semantic_recommendations(user_id, top_k=10):

    user_ids = sorted(interactions_df["user_id"].unique())

    user_index = user_ids.index(user_id)

    user_vector = user_embeddings[user_index]

    scores = cosine_similarity(user_vector.reshape(1, -1), item_embeddings)[0]

    consumed = interactions_df[interactions_df["user_id"] == user_id][
        "item_id"
    ].tolist()

    results = []

    ranked_idx = np.argsort(scores)[::-1]

    for idx in ranked_idx:

        item_id = profiles_df.iloc[idx]["item_id"]

        if item_id in consumed:

            continue

        results.append(
            {
                "item_id": item_id,
                "title": profiles_df.iloc[idx]["title"],
                "semantic_score": scores[idx],
            }
        )

        if len(results) >= top_k:

            break

    return pd.DataFrame(results)

In [18]:
semantic_recommendations(101)

,item_id,title,semantic_score
0,8,Startup Fundraising Guide,0.736429
1,7,Machine Learning in Finance,0.415184
2,6,Modern Data Engineering,0.401263
3,9,Nutrition Science,0.362029
4,4,The Psychology of Habits,0.249460
5,10,Large Language Models,0.244284
6,3,AI for Healthcare,0.230878


In [ ]:
def hybrid_recommendations(
    user_id, top_k=10, collaborative_weight=0.5, semantic_weight=0.5
):

    collab = collaborative_recommendations(user_id, top_k=20)

    semantic = semantic_recommendations(user_id, top_k=20)

    scores = {}

    for _, row in collab.iterrows():

        item_id = row["item_id"]

        scores.setdefault(item_id, 0)

        scores[item_id] += collaborative_weight * row["collaborative_score"]

    for _, row in semantic.iterrows():

        item_id = row["item_id"]

        scores.setdefault(item_id, 0)

        scores[item_id] += semantic_weight * row["semantic_score"]
        results = []

    for item_id, score in scores.items():

        item = profiles_df[profiles_df["item_id"] == item_id].iloc[0]

        results.append(
            {"item_id": item_id, "title": item["title"], "hybrid_score": score}
        )

    results = pd.DataFrame(results)

    return results.sort_values(by="hybrid_score", ascending=False).head(top_k)

In [22]:
hybrid_recommendations(101)

,item_id,title,hybrid_score
0,8,Startup Fundraising Guide,1.556002
1,6,Modern Data Engineering,0.590070
2,7,Machine Learning in Finance,0.519143
4,9,Nutrition Science,0.181014
3,4,The Psychology of Habits,0.124730
6,10,Large Language Models,0.122142
5,3,AI for Healthcare,0.115439


In [23]:
hybrid_recommendations(102)

,item_id,title,hybrid_score
0,5,Cloud Computing Fundamentals,1.103536
3,9,Nutrition Science,0.198426
1,6,Modern Data Engineering,0.149654
2,4,The Psychology of Habits,0.131494
4,7,Machine Learning in Finance,-0.102743
6,10,Large Language Models,-0.293031
5,3,AI for Healthcare,-0.304124


In [24]:
print("Collaborative")

display(collaborative_recommendations(101))

print()

print("Semantic")

display(semantic_recommendations(101))

print()

print("Hybrid")

display(hybrid_recommendations(101))

Collaborative


,item_id,title,collaborative_score
0,8,Startup Fundraising Guide,2.375574e+00
1,6,Modern Data Engineering,7.788767e-01
2,7,Machine Learning in Finance,6.231013e-01
3,4,The Psychology of Habits,2.805968e-17
4,9,Nutrition Science,2.805968e-17
5,3,AI for Healthcare,-7.882155e-16
6,10,Large Language Models,-7.882155e-16



Semantic


,item_id,title,semantic_score
0,8,Startup Fundraising Guide,0.736429
1,7,Machine Learning in Finance,0.415184
2,6,Modern Data Engineering,0.401263
3,9,Nutrition Science,0.362029
4,4,The Psychology of Habits,0.249460
5,10,Large Language Models,0.244284
6,3,AI for Healthcare,0.230878



Hybrid


,item_id,title,hybrid_score
0,8,Startup Fundraising Guide,1.556002
1,6,Modern Data Engineering,0.590070
2,7,Machine Learning in Finance,0.519143
4,9,Nutrition Science,0.181014
3,4,The Psychology of Habits,0.124730
6,10,Large Language Models,0.122142
5,3,AI for Healthcare,0.115439


In [25]:
hybrid_df = hybrid_recommendations(101, top_k=20)

hybrid_df

,item_id,title,hybrid_score
0,8,Startup Fundraising Guide,1.556002
1,6,Modern Data Engineering,0.590070
2,7,Machine Learning in Finance,0.519143
4,9,Nutrition Science,0.181014
3,4,The Psychology of Habits,0.124730
6,10,Large Language Models,0.122142
5,3,AI for Healthcare,0.115439


In [26]:
recommendation_results = hybrid_df.copy()

recommendation_results["user_id"] = 101

recommendation_results

,item_id,title,hybrid_score,user_id
0,8,Startup Fundraising Guide,1.556002,101
1,6,Modern Data Engineering,0.590070,101
2,7,Machine Learning in Finance,0.519143,101
4,9,Nutrition Science,0.181014,101
3,4,The Psychology of Habits,0.124730,101
6,10,Large Language Models,0.122142,101
5,3,AI for Healthcare,0.115439,101


In [27]:
all_recommendations = []

for user_id in sorted(interactions_df["user_id"].unique()):

    user_results = hybrid_recommendations(user_id, top_k=5)

    user_results["user_id"] = user_id

    all_recommendations.append(user_results)

In [28]:
hybrid_df = pd.concat(all_recommendations)

hybrid_df.head()

,item_id,title,hybrid_score,user_id
0,8,Startup Fundraising Guide,1.556002,101
1,6,Modern Data Engineering,0.590070,101
2,7,Machine Learning in Finance,0.519143,101
4,9,Nutrition Science,0.181014,101
3,4,The Psychology of Habits,0.124730,101


In [29]:
hybrid_df["item_id"].nunique()

10

In [30]:
hybrid_df.groupby("user_id").size()

user_id
101    5
102    5
103    5
104    5
105    5
dtype: int64

In [31]:
hybrid_df.to_csv("../data/hybrid_recommendations.csv", index=False)

In [32]:
saved_df = pd.read_csv("../data/hybrid_recommendations.csv")

saved_df.head()

,item_id,title,hybrid_score,user_id
0,8,Startup Fundraising Guide,1.556002,101
1,6,Modern Data Engineering,0.590070,101
2,7,Machine Learning in Finance,0.519143,101
3,9,Nutrition Science,0.181014,101
4,4,The Psychology of Habits,0.124730,101
